In [11]:
# Installing Packages for Project - can use % instead of ! if you have any installation issues
!pip install pandas
!pip install numpy
%pip install torch --index-url https://download.pytorch.org/whl/cpu #Downloads CPU based Torch - not full CUDA Libraries aswell
!pip install matplotlib
!pip install scikit-learn
%pip install polars
%pip install pyarrow

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 4.7 MB/s  0:00:11 eta 0:00:010:01:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Importing all packages into the project
import pandas as pd
import numpy as np
import polars as pl
import sklearn
import matplotlib.pyplot as plt
import torch
import pyarrow

In [3]:
# Parser to read out all Raw Time-Series Data for each user
# Four subdirectories exist under the main raw folder so we go through all 
# Concat each file to the df, we get the Sensor & Device from the filename regex
# Also use filename regex filter to avoid the .DS_Store files

import re 
from pathlib import Path

RAW_DIRECTORY = Path("/home/busing/Documents/Projects/NCP-Wearables/wisdm+smartphone+and+smartwatch+activity+and+biometrics+dataset/wisdm-dataset/raw/") #File path for dataset on device
FILE_NAME = re.compile(r"data_(\d+)_(accel|gyro)_(phone|watch)\.txt$")

print(RAW_DIRECTORY.exists()) # Print confirmation that Notebook can find the filepath
print(len(list(RAW_DIRECTORY.rglob("*.txt")))) # Should print 204 as this is # of .txt files with raw user data

def parse_file(path):
    m = FILE_NAME.search(path.name)
    df = pd.read_csv(path, header = None, 
                    names=["Subject_ID", "Activity_Code", "Timestamp", "x", "y", "z"], 
                     dtype={"Subject_ID": "int", "Activity_Code": "category", "Timestamp": "int64",
                            "x": "float32", "y": "float32", "z": "string"})
    df["z"] = df["z"].str.rstrip(";").astype("float32")
    df["Device"], df["Sensor"] = m.group(3), m.group(2)
    return df

def load_WISDM(raw_dir=RAW_DIRECTORY):
    files = sorted(p for p in Path(raw_dir).rglob("*.txt") if FILE_NAME.search(p.name))
    df = pd.concat([parse_file(p) for p in files], ignore_index = True)
    for c in ("Activity_Code", "Device", "Sensor"): 
        df[c] = df[c].astype("category")
    return df[["Subject_ID", "Activity_Code", "Timestamp", "Device", "Sensor", "x", "y", "z"]]

df = load_WISDM()

df.to_parquet("wisdm_raw.parquet") #Write data to parquet file so we do not need parse data with every kernel restart.


True
204


In [2]:
# Read in data from Parquet File that was created by the parser.
# Much quicker to load in data than parsing with Pandas on each kernel restart.
df = pd.read_parquet("/home/busing/Documents/Projects/NCP-Wearables/wisdm_raw.parquet")
print(df.head()) # Confirm data loaded in after kernel reset

   Subject_ID Activity_Code        Timestamp Device Sensor         x  \
0        1600             A  252207666810782  phone  accel -0.364761   
1        1600             A  252207717164786  phone  accel -0.879730   
2        1600             A  252207767518790  phone  accel  2.001495   
3        1600             A  252207817872794  phone  accel  0.450623   
4        1600             A  252207868226798  phone  accel -2.164352   

           y         z  
0   8.793503  1.055084  
1   9.768784  1.016998  
2  11.109070  2.619156  
3  12.651642  0.184555  
4  13.928436 -4.422485  


In [4]:
# Setup "stream" as one device + sensor combination thus there are 4 per participant
# Stream joing Device & Sensor into one label like e.g phone_accel. 
# Since this follows the filename format it allows key to identify each indivdual file from the dataset
KEY = ["Subject_ID", "Stream"]

df["Stream"] = (df["Device"].astype(str) + "_" + df["Sensor"].astype(str)).astype("category")
df["Row_Order"] = df.groupby(KEY, observed = True).cumcount().astype("int32") # Add int counter per file, gives us something to sort by other than timestamp for segmentation

ACTIVITY = {"A": "Walking", "B": "Jogging", "C": "Stairs", "D": "Sitting", "E": "Standing",
            "F": "Typing", "G": "Brushing Teeth", "H": "Eating Soup", "I": "Eating Chips", "J": "Eating Pasta", "K": "Drinking from Cup",
            "L": "Eating Sandwich", "M": "Kicking Soccerball", "O": "Playing Catch with Tennis Ball", "P": "Dribbling Basketball", "Q": "Writing",
            "R": "Clapping", "S": "Folding Clothes"}

EXPECTED_USERS = list(range(1600, 1651))                
EXPECTED_STREAMS = ["phone_accel", "phone_gyro", "watch_accel", "watch_gyro"]
EXPECTED_CODES = list(ACTIVITY)                         

#Check we get the expected count output, should match dataset
print(f"rows: {len(df):,} | participants: {df['Subject_ID'].nunique()} | "
      f"activities: {df['Activity_Code'].nunique()} | streams: {df['Stream'].nunique()}")

rows: 15,630,426 | participants: 51 | activities: 18 | streams: 4


In [10]:
# Count by Participant, Activity & Stream

#Note observed = True ensure we are only getting the combinations that exist in the dataframe not new combinations that are not actually present. 

# Rows per participant per stream
by_user_stream = df.groupby(KEY, observed=True).size().unstack("Stream")
#Creates a table w/ participants as rows and streams as cols. 
#Values are the number of sensor readings per participant
display(by_user_stream)

display(by_user_stream.describe()) # Provides basic summary stats for each stream

# Rows per activity; Col per stream
# .rename(index=ACTIVITY) shows activity w/ name as opposed to with Activity code
by_act_stream = (df.groupby(["Activity_Code", "Stream"], observed=True).size().unstack("Stream").rename(index=ACTIVITY))
display(by_act_stream)

# Rows per participant; Cols per activity (all streams combined)
by_user_act = (df.groupby(["Subject_ID", "Activity_Code"], observed=True).size()
                 .unstack("Activity_Code"))
display(by_user_act)

Stream,phone_accel,phone_gyro,watch_accel,watch_gyro
Subject_ID,,,,
1600,64311,64247,65462,65435
1601,81457,81193,64840,64829
1602,84890,64286,64985,64950
1603,81841,81199,64865,64851
1604,64314,64243,64904,64870
1605,81281,81181,66195,66187
1606,64315,64243,64890,64854
1607,76702,76689,65830,65810
1608,96523,64249,68198,68156


Stream,phone_accel,phone_gyro,watch_accel,watch_gyro
count,51.000000,51.000000,51.000000,51.000000
mean,94203.980392,70757.549020,74059.725490,67457.686275
std,36177.855738,11300.882807,25962.081637,9352.135727
min,60771.000000,57106.000000,60156.000000,60119.000000
25%,64314.500000,64246.000000,64881.500000,64854.000000
50%,81459.000000,64336.000000,64906.000000,64871.000000
75%,112938.000000,80347.000000,67250.000000,67215.000000
max,163768.000000,128500.000000,160799.000000,129738.000000


Stream,phone_accel,phone_gyro,watch_accel,watch_gyro
Activity_Code,,,,
Walking,279817,203919,210495,192531
Jogging,268409,200252,205787,187833
Stairs,255645,197857,207312,180416
Sitting,264592,202370,213018,195050
Standing,269604,202351,216529,194103
Typing,246356,194540,205137,187175
Brushing Teeth,269609,202622,208720,190759
Eating Soup,270756,202408,209483,187057
Eating Chips,261360,197905,210048,192085


Activity_Code,A,B,C,D,E,F,G,H,I,J,K,L,M,O,P,Q,R,S
Subject_ID,,,,,,,,,,,,,,,,,,
1600,14351.0,14350.0,14351.0,14352.0,14350.0,14351.0,14349.0,14349.0,14349.0,14349.0,14352.0,14354.0,14351.0,14349.0,14352.0,14352.0,14351.0,15493.0
1601,16227.0,16227.0,16223.0,16225.0,16227.0,16225.0,16223.0,16228.0,16227.0,16225.0,16222.0,16225.0,16230.0,16234.0,16227.0,16476.0,16223.0,16225.0
1602,14349.0,14349.0,14350.0,14349.0,14542.0,14351.0,14351.0,19729.0,18730.0,19734.0,18888.0,14351.0,14349.0,14349.0,14349.0,14349.0,14350.0,15292.0
1603,16226.0,16229.0,16229.0,16478.0,16226.0,16226.0,16227.0,16227.0,16226.0,16261.0,16479.0,16228.0,16356.0,16230.0,16225.0,16225.0,16229.0,16229.0
1604,14350.0,14351.0,14351.0,14349.0,14350.0,14376.0,14350.0,14349.0,14352.0,14351.0,14351.0,14351.0,14350.0,14351.0,14350.0,14349.0,14349.0,14351.0
1605,16228.0,16226.0,16225.0,16226.0,16220.0,16225.0,16259.0,16224.0,18891.0,16227.0,16221.0,16223.0,16224.0,16226.0,16321.0,16225.0,16229.0,16224.0
1606,14349.0,14349.0,14350.0,14351.0,14350.0,14350.0,14350.0,14352.0,14350.0,14352.0,14349.0,14349.0,14349.0,14351.0,14350.0,14350.0,14351.0,14350.0
1607,16232.0,16229.0,16236.0,16227.0,16227.0,16227.0,16226.0,16230.0,16226.0,7205.0,16225.0,18177.0,16227.0,16228.0,16231.0,16227.0,16226.0,16225.0
1608,19718.0,19718.0,14352.0,14349.0,14349.0,14349.0,14351.0,14362.0,14349.0,20952.0,14348.0,19724.0,19718.0,19718.0,19718.0,14352.0,14349.0,14350.0


In [12]:
# Investigating Missing Files & Missing Activities

present = set(df[KEY].astype({"Stream": str}).drop_duplicates().itertuples(index=False, name=None))
# List Comprehension to do Loop to check for any missing files u = users s = streams
missing_files = [(u, s) for u in EXPECTED_USERS for s in EXPECTED_STREAMS if (u,s) not in present]
print(f"files present: {len(present)} / {len(EXPECTED_USERS) * len(EXPECTED_STREAMS)} ; Missing Files: {missing_files}")

# Expected - Every file has all 18 activities
combo = (df.groupby(KEY + ["Activity_Code"], observed = True).size().rename("rows").reset_index()
        .astype({"Stream": str, "Activity_Code": str}))
# Set of (participant, stream, activity) triples that actually exist in the dataframe
df_has = set(combo[["Subject_ID", "Stream", "Activity_Code"]].itertuples(index=False, name=None))

# The list comprehension builds all 51 x 4 x 18 expected triples and compares to df_has which provides us with the gaps.
missing_activities = pd.DataFrame(
    [(u, s, c, ACTIVITY[c]) for u in EXPECTED_USERS for s in EXPECTED_STREAMS for c in EXPECTED_CODES
     if (u, s, c) not in df_has],
    columns=["Subject_ID", "Stream", "Activity_Code", "Activity"])

print(f"Missing (participant, stream, activity) combinations: {len(missing_activities)}")

display(missing_activities)

files present: 204 / 204 ; Missing Files: []
Missing (participant, stream, activity) combinations: 35


,Subject_ID,Stream,Activity_Code,Activity
0,1607,phone_accel,J,Eating Pasta
1,1607,phone_gyro,J,Eating Pasta
2,1609,phone_accel,B,Jogging
3,1609,phone_gyro,B,Jogging
4,1616,phone_accel,B,Jogging
5,1616,phone_accel,F,Typing
6,1616,phone_gyro,B,Jogging
7,1616,phone_gyro,F,Typing
8,1616,watch_accel,B,Jogging
9,1616,watch_gyro,B,Jogging


In [13]:
# Investigate Duplicate Records & Timestamps

# keep = first; so that only Second and later copies of a record are flagged as duplicates

# Returns True for any row whose timestape already appeared earlier in the same file
dup_ts = df.duplicated(subset=KEY + ["Timestamp"], keep="first")
# Returns True when a whole row repeats - same activity, timestamp, x, y, z reading
dup_exact = df.duplicated(subset=KEY + ["Activity_Code", "Timestamp", "x", "y", "z"], keep="first") # identical row
# Returns True when timestamp is shared in the same file but the readings (x, y, z) are different
dup_conflict = dup_ts & ~dup_exact 

print(f"duplicate timestamps (extra copies): {dup_ts.sum():,}  ({dup_ts.mean():.2%} of rows)")
print(f"exact duplicate rows: {dup_exact.sum():,}")
print(f"same timestamp, different values: {dup_conflict.sum():,}")

# Lists which files contain duplicates and how many extra rows exist in that file.
dup_summary = (df[dup_ts].groupby(KEY, observed=True).size().rename("extra_rows").reset_index()
                 .query("extra_rows > 0"))
print(f"\nfiles affected: {len(dup_summary)} / {df.groupby(KEY, observed=True).ngroups}")
display(dup_summary)

duplicate timestamps (extra copies): 266,015  (1.70% of rows)
exact duplicate rows: 263,696
same timestamp, different values: 2,319

files affected: 6 / 204


,Subject_ID,Stream,extra_rows
0,1629,phone_accel,69673
1,1629,phone_gyro,64250
2,1629,watch_accel,64904
3,1629,watch_gyro,64869
4,1649,watch_accel,1160
5,1649,watch_gyro,1159


In [35]:
# Investigating Timestamps and resets

g = df.groupby(KEY, observed = True) #G represents one file as KEY indicates a Sensor, Device Tuple just as each file does
dt_raw = g["Timestamp"].diff() #Difference between each timestamp and Prev timestamp in file

# True on the first row of each activity block. Compares each row's activity with previous rows activity
new_run = df["Activity_Code"].ne(g["Activity_Code"].shift())  # Find where Activity Changes

# As First Row of New Activity Block occurs during file and not just first line, we need to find the boundary. We also check this point has a valid Timestamp
boundary = new_run & dt_raw.notna()

neg = dt_raw < 0 # Clock Resets

print(f"Number of long gaps: {long_gaps.sum()}")

print(f"backwards steps: {neg.sum():,}")
print(f"at an activity boundary: {(neg & boundary).sum():,}")
print(f"inside an activity: {(neg & ~boundary).sum():,}")
print(f"zero steps (repeat timestamp): {(dt_raw == 0).sum():,}")

# Is every activity one contiguous block per file? (more than one block = recording replayed/split)
blocks = new_run.groupby([df["Subject_ID"], df["Stream"], df["Activity_Code"]], observed=True).sum()
print(f"\n(participant, stream, activity) with more than one block: {(blocks > 1).sum()}")
display(blocks[blocks > 1].reset_index(name="blocks").groupby(["Subject_ID", "Stream"], observed=True).size()
        .rename("activities_split").reset_index())


Number of long gaps: 2058
backwards steps: 1,457
at an activity boundary: 1,457
inside an activity: 0
zero steps (repeat timestamp): 0

(participant, stream, activity) with more than one block: 72


,Subject_ID,Stream,activities_split
0,1629,phone_accel,18
1,1629,phone_gyro,18
2,1629,watch_accel,18
3,1629,watch_gyro,18


In [15]:
# Investigating Sampling Rate we actually see in the data & Block Lenghts

d = df.loc[~dup_ts].copy() # Dropped duplicate timestamps so duplicates do not mess with results
g = d.groupby(KEY, observed = True) # as above, just on non-duplicated timestamps
d["new_run"] = d["Activity_Code"].ne(g["Activity_Code"].shift())
d["run_id"] = d.groupby(KEY, observed=True)["new_run"].cumsum() # number of activity block in each file
RUN = KEY + ["run_id"]
d["dt_ms"] = d.groupby(RUN, observed=True)["Timestamp"].diff() / 1e6 # step between samples in m/s within a block (does not measure gap between blocks/activities)

#
runs = (d.groupby(RUN, observed=True).agg(Activity_Code=("Activity_Code", "first"),
                                          n=("Timestamp", "size"), 
                                          t0=("Timestamp", "min"),
                                          t1=("Timestamp", "max")).reset_index())

runs["duration_s"]= (runs["t1"] - runs["t0"]) / 1e9
runs["sampling_rate_hz"] = (runs["n"] - 1) / runs["duration_s"]
runs["rate_class"] = (runs["sampling_rate_hz"] / 5).round() * 5  #nearest 5 Hz

print(f"activity blocks: {len(runs)}  (expected {len(EXPECTED_USERS)*4*18} if complete)")
# Summary Statistics - We should see Central tendacy close to 180s (3 mins) as described in dataset description
display(runs["duration_s"].describe().round(1).to_frame("duration_s (expected ~180)"))
# Displays how many blocks fall into each rate class per stream. Should see most around 20 Hz
display(runs.groupby(["Stream", "rate_class"], observed=True).size().unstack("rate_class", fill_value=0))


activity blocks: 3637  (expected 3672 if complete)


,duration_s (expected ~180)
count,3637.0
mean,179.9
std,4.2
min,121.8
25%,179.8
50%,179.8
75%,179.9
max,351.3


rate_class,10.0,20.0,25.0,30.0,35.0,40.0,45.0,50.0
Stream,,,,,,,,
phone_accel,0,495,166,1,3,2,3,240
phone_gyro,0,589,317,0,1,0,0,1
watch_accel,1,817,7,8,5,1,2,71
watch_gyro,1,816,73,8,5,1,2,1


In [25]:
# Activity Blocks where Length is significantly differnet more than 2 S.D from expected 3 mins

short_long_blocks = runs[(runs["duration_s"] < 170) | (runs["duration_s"] > 190)]
print(f"Number of blocks 2 standard deviations outside 3 mins: {len(short_long_blocks)}")

#Print a table of all activity blocks where the time is more than 2 S.D from 3 mins
display(short_long_blocks.assign(Activity=lambda t: t["Activity_Code"].astype(str).map(ACTIVITY))
        [KEY + ["Activity", "n", "duration_s", "sampling_rate_hz"]].round(1))


Number of blocks 2 standard deviations outside 3 mins: 4


,Subject_ID,Stream,Activity,n,duration_s,sampling_rate_hz
3541,1649,watch_accel,Kicking Soccerball,3914,351.3,11.1
3542,1649,watch_accel,Playing Catch with Tennis Ball,2441,121.8,20.0
3559,1649,watch_gyro,Kicking Soccerball,3912,351.2,11.1
3560,1649,watch_gyro,Playing Catch with Tennis Ball,2441,121.8,20.0


In [20]:
# Further Investigation of Sampling Rates outside 20 Hz

# Sampling rate per participant per stream - Median across Activity
rate_by_file = runs.groupby(KEY, observed=True)["sampling_rate_hz"].agg(["median", "min", "max"]).round(1)

# Is Hz Median outside Range of 18 Hz - 22 Hz? # Is Hz unstable? Fastest vs Slowest Block Differ by more than 10 Hz
unstable_hz = rate_by_file[(rate_by_file["median"] < 18) | (rate_by_file["median"] > 22) | 
                           (rate_by_file["max"] - rate_by_file["min"] > 10)] 

print(f"files not at a steady 20 Hz : {len(unstable_hz)} / {len(rate_by_file)}")
display(unstable_hz)

files not at a steady 20 Hz : 80 / 204


median   min   max
Subject_ID Stream                         
1601       phone_accel    25.1  25.1  26.5
           phone_gyro     25.1  25.1  25.1
1602       phone_accel    19.9  19.9  49.6
1603       phone_accel    25.1  25.1  26.5
           phone_gyro     25.1  25.1  25.1
...                        ...   ...   ...
1648       phone_gyro     24.8  24.8  24.8
1649       phone_accel    25.1  25.1  26.5
           phone_gyro     25.1  25.1  25.1
1650       phone_accel    49.6  49.6  49.7
           phone_gyro     24.8  24.8  24.8

[80 rows x 3 columns]

In [23]:
# Investigating Timestamp Gaps & Step Sizes within Blocks where Sample Rates Differ

w = d.dropna(subset=["dt_ms"])

# Loop throuhg cut-off provided cut-off sizes (m/s) for each cut-off provided, count the number of steps that longer than this.
print("steps longer than:")
for t_h in (100, 250, 500, 1000, 5000):
    print(f"{t_h: > 5} ms: {(w['dt_ms']> t_h).sum():,}")

long_gaps = (w.loc[w["dt_ms"] > 500, KEY + ["Activity_Code", "dt_ms"]]
            .assign(Activity=lambda t: t["Activity_Code"].astype(str).map(ACTIVITY)))
display(long_gaps)


# We know there are portions of the data where the sampling rate changes significantly in a stream
# investigation of whether this sampling rate change occurs within an activity block as this will affect number of samples in a window

# Get the 5th Percentile and 95th Percentile for each activity
step_q = w.groupby(RUN, observed=True)["dt_ms"].quantile([0.05, 0.95]).unstack()
# Find Activity Blocks where the 95th percentile is 2x greater than the 5th Percentile
mixed = step_q[step_q[0.95] > 2 * step_q[0.05]].reset_index()

#Print Summary and Table of this data
print(f"\nblocks with mixed step sizes (95% > 2 x 5%): {len(mixed)} / {len(step_q)}")
display(mixed.groupby(KEY, observed=True).size().rename("mixed_blocks").reset_index()
        .sort_values("mixed_blocks", ascending=False).head(15))

steps longer than:
  100 ms: 962
  250 ms: 552
  500 ms: 10
 1000 ms: 10
 5000 ms: 8


,Subject_ID,Stream,Activity_Code,dt_ms,Activity
12103811,1649,watch_accel,M,1569.807596,Kicking Soccerball
12103879,1649,watch_accel,M,56652.537154,Kicking Soccerball
12103947,1649,watch_accel,M,56655.903353,Kicking Soccerball
12104018,1649,watch_accel,M,17494.902891,Kicking Soccerball
12104183,1649,watch_accel,M,23844.253068,Kicking Soccerball
15544199,1649,watch_gyro,M,1569.807596,Kicking Soccerball
15544267,1649,watch_gyro,M,56652.537154,Kicking Soccerball
15544335,1649,watch_gyro,M,56655.903353,Kicking Soccerball
15544406,1649,watch_gyro,M,17494.902891,Kicking Soccerball
15544571,1649,watch_gyro,M,23844.253068,Kicking Soccerball



blocks with mixed step sizes (p95 > 2 x p5): 86 / 3637


,Subject_ID,Stream,mixed_blocks
14,1609,watch_gyro,4
13,1609,watch_accel,4
38,1628,phone_accel,4
3,1602,phone_accel,3
32,1625,phone_accel,3
45,1644,watch_gyro,3
44,1644,watch_accel,3
48,1647,phone_accel,3
51,1649,phone_accel,3
28,1621,watch_accel,2


In [29]:
# Sliding Window Size Exploration

WINDOW_SIZE = 10 # Seconds

# For each rate class, how many blocks there are and how many samples a X WINDOW_SIZE window would contain

tbl = (runs.groupby("rate_class")["sampling_rate_hz"].agg(blocks="size", mean_rate="mean").round(1)
       .assign(samples_per_size = lambda t: (t["mean_rate"] * WINDOW_SIZE).round(0)))
display(tbl)

,blocks,mean_rate,samples_per_size
rate_class,,,
10.0,2,11.1,111.0
20.0,2717,20.0,200.0
25.0,563,25.0,250.0
30.0,17,30.7,307.0
35.0,14,34.0,340.0
40.0,4,38.5,385.0
45.0,7,44.5,445.0
50.0,313,49.7,497.0


In [13]:
# Week 2 - Task 1: Segmenetation of Files at Activity Change, Clock Resets and Long Gaps
# Before any within segment sorting. 

# Finding Segmentation Boundaries

GAP_BREAK_S = 1.0 # Step longer than this starts a new segment
RESET_TOLERANCE_S = 1.0 # A backwards step bigger than this is a clock reset

GAP_BREAK_NS = int(GAP_BREAK_S * 1e9) # Timestamps are in Nanoseconds so converting to Nanoseconds
RESET_TOLERANCE_NS = int(RESET_TOLERANCE_S * 1e9)

clone_df = df.copy() # COPY DATAFRAME - We want to leave raw data untouched; can always create a new Parquet file with Segemented & Windowed Data

timestamp = clone_df["Timestamp"].to_numpy(dtype = "int64")
subject_id = clone_df["Subject_ID"].to_numpy()
stream = clone_df["Stream"].cat.codes.to_numpy() 
activity = clone_df["Activity_Code"].cat.codes.to_numpy()

n = len(clone_df)


#Find the step between each row and the reading before
dt = np.zeros(n, dtype="int64") # Create an array of 0's the length of dataset
dt[1:] = timestamp [1:] - timestamp[:-1] #

# Each flag is True on the First row of a new segment

new_file = np.ones(n, dtype=bool) # Row 0 is always a new file
new_file[1:] = (subject_id[1:] != subject_id[:-1]) | (stream[1:] != stream[:-1])

new_activity = np.zeros(n, dtype = bool) 
new_activity[1:] = activity[1:] != activity [:-1] 

clock_reset = dt < -RESET_TOLERANCE_NS
long_gap = dt > GAP_BREAK_NS

breaks = new_file | new_activity | clock_reset | long_gap # Breaks are anything that matches one of our flags

# Sum of all break flags 
clone_df["Seg_ID"] = np.cumsum(breaks) - 1 # -1 as 0 index

print(f"segments: {clone_df['Seg_ID'].nunique()}")
print(f"breaks from new file: {new_file.sum()}")
print(f"from activity change: {(new_activity & ~new_file).sum()}")
print(f"from clock resets: {clock_reset.sum()}")
print(f"from long gaps: {long_gap.sum()}")


segments: 3719
breaks from new file: 204
from activity change: 3505
from clock resets: 1568
from long gaps: 2150


In [14]:
# Sort within Segments & Resolve Duplicate Timestamps

clone_df = clone_df.sort_values(["Seg_ID", "Timestamp"], kind = "stable") #Stable means ties keep order they appear in the file

K = ["Seg_ID", "Timestamp"] # Duplicate = same timestamp inside the same segement

dup_all = clone_df.duplicated(K, keep=False) # Flag every row that is a duplicate including the first occurence, different to how we calculated before. 
dup_rows = clone_df.loc[dup_all, K + ["x", "y", "z"]] # List of all rows that have

distinct = dup_rows.drop_duplicates()
n_distinct = distinct.groupby(K).size()
conflict_keys = n_distinct[n_distinct > 1].index

# Log all conflicting duplicate rows to csv file as requested
conflict_log = dup_rows.merge(conflict_keys.to_frame(index=False), on = K)
conflict_log.to_csv("conflict_log.csv", index= False)

In [16]:
# Drop Duplicates from Cloned Dataframe
clone_df = clone_df[~clone_df.duplicated(K, keep="first")] # Keep the first occurence of a timestamp, remove the rest. 

In [17]:
# Creating segment table and dropping replayed copies in Raw Time-Series Data

seg = (clone_df.groupby("Seg_ID")
         .agg(Subject_ID=("Subject_ID", "first"),
              Stream=("Stream", "first"),
              Activity_Code=("Activity_Code", "first"),
              n=("Timestamp", "size"),
              t0=("Timestamp", "min"),        # integer in nanoseconds, kept as int64
              t1=("Timestamp", "max"),
              x_sum=("x", "sum")))            # cheap fingerprint of the values
seg["Stream"] = seg["Stream"].astype(str)
seg["Activity_Code"] = seg["Activity_Code"].astype(str)

# A replayed segment has the same signature as an earlier one in the same file. 
# Signature is Subject_ID, Steam (Device, Sensor), Activity_Code, Min-Timestamp, Max-Timestamp, 
SIG = ["Subject_ID", "Stream", "Activity_Code", "t0", "t1", "n", "x_sum"]
replay = seg.duplicated(SIG, keep="first")
print(f"replayed (exact-copy) segments dropped: {replay.sum()}")
print(seg[replay].groupby(["Subject_ID", "Stream"]).size())     # expect mostly subject 1629

clone_df = clone_df[~clone_df["Seg_ID"].isin(seg.index[replay])]
seg = seg[~replay].copy() 

# occurrence number of each activity within each file (0 = first time, 1 = second time...)
seg["occ"] = seg.groupby(["Subject_ID", "Stream", "Activity_Code"]).cumcount()
seg[["Device", "Sensor"]] = seg["Stream"].str.split("_", expand=True)

print(f"segments kept: {len(seg):,}")
print(f"(subject, stream, activity) with >1 occurrence: {(seg['occ'] > 0).sum()}")

replayed (exact-copy) segments dropped: 72
Subject_ID  Stream     
1629        phone_accel    18
            phone_gyro     18
            watch_accel    18
            watch_gyro     18
dtype: int64
segments kept: 3,647
(subject, stream, activity) with >1 occurrence: 10


In [19]:
# Week 2 - Task 2: Use Common Time Origin for Accelerometer and Gyro After Verifiying their Clock Relationship
# Propose within device validity rules - at least 160 samples per sensor per 10 second window, observed span >= 8s and no gaps above 0.5 s
# Check these choices on training records and freeze them. 
# Compute statisitics directly on retained samples; no interpolation.

# Check if the accelerometer and gyro on the same clock? 
CLOCK_TOL_S = 1.0       # accel/gyro starts must agree within this to count as "same clock"

PAIR_KEY = ["Subject_ID", "Device", "Activity_Code", "occ"]
acc = seg[seg["Sensor"] == "accel"].reset_index().rename(columns={"Seg_ID": "seg_id_acc"})
gyr = seg[seg["Sensor"] == "gyro" ].reset_index().rename(columns={"Seg_ID": "seg_id_gyr"})

pairs = acc.merge(gyr, on=PAIR_KEY, suffixes=("_acc", "_gyr"), how="outer", indicator=True)
print(pairs["_merge"].value_counts())            # 'both' is what we want; others = unpaired segments

pairs = pairs[pairs["_merge"] == "both"].copy()

# Subtract as int64 FIRST, then convert to seconds this is done as it keeps precision
pairs["start_off_s"] = (pairs["t0_gyr"] - pairs["t0_acc"]) / 1e9
pairs["end_off_s"]   = (pairs["t1_gyr"] - pairs["t1_acc"]) / 1e9

print(pairs[["start_off_s", "end_off_s"]].abs().describe().round(4))

# Pairs that disagree are not trusted to share a clock; excluding them
bad = pairs[(pairs["start_off_s"].abs() > CLOCK_TOL_S) | (pairs["end_off_s"].abs() > CLOCK_TOL_S)]
print(f"pairs failing the clock check: {len(bad)}")
display(bad[PAIR_KEY + ["start_off_s", "end_off_s"]])
pairs = pairs.drop(bad.index).reset_index(drop=True)

_merge
both          1820
left_only        7
right_only       0
Name: count, dtype: int64
       start_off_s  end_off_s
count    1820.0000  1820.0000
mean        0.5973     0.5024
std        20.1696    20.1739
min         0.0000     0.0000
25%         0.0495     0.0000
50%         0.0495     0.0499
75%         0.1511     0.0499
max       860.5536   860.6776
pairs failing the clock check: 25


,Subject_ID,Device,Activity_Code,occ,start_off_s,end_off_s
583,1616,phone,L,0,1.397614,0.016785
584,1616,phone,M,0,1.280121,-0.008942
585,1616,phone,O,0,1.202911,-0.025726
586,1616,phone,P,0,1.350616,-0.019012
716,1620,phone,D,0,1.206268,-0.005585
719,1620,phone,G,0,1.152557,-0.016785
720,1620,phone,H,0,1.165968,-0.039154
721,1620,phone,I,0,1.035065,0.001129
730,1620,phone,S,0,1.582245,-0.008942
897,1625,phone,E,0,1.152557,-0.019012


In [20]:
# -Find common origin per accel/gyro pair 
pairs["pair_id"] = np.arange(len(pairs))
pairs["origin_ns"] = np.minimum(pairs["t0_acc"], pairs["t0_gyr"])     # dtype = int64

# Seg_ID to origin_ns lookup, covering both the accel and gyro segment of every pair
seg_origin = pd.concat([pairs.set_index("seg_id_acc")["origin_ns"],
                        pairs.set_index("seg_id_gyr")["origin_ns"]])

# Keep only rows that belong to a verified pair. Do this BEFORE mapping so there are no NaNs,
# because NaNs would silently turn the int64 origin into float64 and lose precision.
clone_df = clone_df[clone_df["Seg_ID"].isin(seg_origin.index)]
origin = clone_df["Seg_ID"].map(seg_origin).astype("int64")

# int64 subtract first, THEN divide: exact small integers -> float seconds
clone_df["t_s"] = (clone_df["Timestamp"] - origin) / 1e9

clone_df = clone_df.sort_values(["Seg_ID", "t_s"], kind="stable")     # (already sorted, cheap safety)

print(clone_df.head())

   Subject_ID Activity_Code        Timestamp Device Sensor         x  \
0        1600             A  252207666810782  phone  accel -0.364761   
1        1600             A  252207717164786  phone  accel -0.879730   
2        1600             A  252207767518790  phone  accel  2.001495   
3        1600             A  252207817872794  phone  accel  0.450623   
4        1600             A  252207868226798  phone  accel -2.164352   

           y         z       Stream  Row_Order  Seg_ID       t_s  
0   8.793503  1.055084  phone_accel          0       0  0.000000  
1   9.768784  1.016998  phone_accel          1       0  0.050354  
2  11.109070  2.619156  phone_accel          2       0  0.100708  
3  12.651642  0.184555  phone_accel          3       0  0.151062  
4  13.928436 -4.422485  phone_accel          4       0  0.201416  


In [23]:
# Windows & Per Sensor Statistics

WINDOW_S = 10.0
STEP_S = 5.0 

t = clone_df["t_s"].to_numpy()
seg_arr = clone_df["Seg_ID"].to_numpy()

# Row Range of every segement inside the sorted array
first = np.flatnonzero(np.r_[True, seg_arr[1:] != seg_arr[:-1]]) #Find the starting indices of consecutive segments where values change
last     = np.r_[first[1:], len(seg_arr)]
bounds   = dict(zip(seg_arr[first], zip(first, last)))


def window_stats(ts_seg, starts): 
    low = np.searchsorted(ts_seg, starts, side = 'left')
    high = np.searchsorted(ts_seg, starts + WINDOW_S, side = "left")
    cnt = high - low
    span = np.zeros(len(starts))
    gap = np.full(len(starts), np.inf)
    for i, (a,b) in enumerate (zip(low, high)): 
        if b - a >= 2: 
            s = ts_seg[a:b]
            span[i] = s[-1] - s[0]
            gap[i] = np.diff(s).max()
    return low, high, cnt, span, gap

parts = []

for p in pairs.itertuples():
    a0, a1 = bounds[p.seg_id_acc]
    g0, g1 = bounds[p.seg_id_gyr]
    ta, tg = t[a0:a1], t[g0:g1]
    t_end = max(ta[-1], tg[-1])
    starts = np.arange(0.0, t_end - WINDOW_S + 1e-9, STEP_S)
    if len(starts) == 0:
        continue
    la, ha, na, spa, gpa = window_stats(ta, starts)
    lg, hg, ng, spg, gpg = window_stats(tg, starts)
    parts.append(pd.DataFrame({
            "pair_id" : p.pair_id, "Subject_ID" : p.Subject_ID, "Device": p.Device,
            "Activity_Code": p.Activity_Code, "occ": p.occ, "t_start": starts,
            "acc_a": a0 + la, "acc_b": a0 + ha, "gyr_a": g0 + lg, "gyr_b": g0 + hg, 
            "acc_n": na, "acc_span": spa, "acc_gap": gpa, "gyr_n": ng, "gyr_span": spg, "gyr_gap": gpg}))

windows = pd.concat(parts, ignore_index = True)
print(f"Number of candidate windows: {len(windows)}")

Number of candidate windows: 61048
